# Web-Gold-40K - recovery v2.7 full controlled workflow

This is the complete audit, smoke, bbox-overfit, diagnostic, and five-epoch mini workflow. V2.7 keeps the v2.6 data, model, loss, optimizer, and thresholds unchanged. Its only experimental change is checkpoint selection: first require every registered quality gate, then rank eligible epochs by outcome MCC.

The successful no-GPU replay is preserved separately in `kaggle_gold_recovery_v2_7_selection.ipynb`. Because all structural gates already passed under the identical v2.6 training path, the default here is `mini` for a fresh physical v2.7 checkpoint and round-trip. Set another stage only when independently repeating that gate. Restart the kernel before changing stages. The test split stays locked in every stage.

In [ ]:
# 1. Pull the latest modular code and record the environment.
from pathlib import Path
import csv
import importlib.metadata as metadata
import json, os, subprocess, sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
environment = {name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
environment_path = Path('/kaggle/working/gold_recovery_v2_7_environment.json')
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))

In [ ]:
# 2. Locate the attached dataset without downloading or extracting it.
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')

def find_split_root(root: Path) -> Path:
    if all((root / name).is_file() for name in SPLIT_FILES):
        return root
    candidates = []
    for current, _, files in os.walk(root, followlinks=True):
        if set(SPLIT_FILES).issubset(files):
            candidates.append(Path(current))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one structured split folder; found {candidates}')
    return candidates[0]

DATA_ROOT = find_split_root(ATTACHED_ROOT).resolve()
print('DATA_ROOT =', DATA_ROOT)

In [ ]:
# 3. Select exactly one gate. Restart the kernel before changing stages.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed

STAGE = 'mini'  # 'audit', 'smoke', 'bbox_overfit', 'diagnostic', or 'mini'
BBOX_MONTAGE_REVIEWED = True  # Keep True only if all 32 green target boxes were reviewed.
SEED = 42
TRAIN_ROWS, VAL_ROWS = 5_000, 500
EPOCHS = 1 if STAGE == 'diagnostic' else 5
ALLOWED_STAGES = {'audit', 'smoke', 'bbox_overfit', 'diagnostic', 'mini'}
assert STAGE in ALLOWED_STAGES
if STAGE != 'audit':
    assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)

cfg = load_config('configs/backbones/qwen2vl_2b_gold_v2_7.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['num_workers'] = 0 if STAGE in {'audit', 'smoke', 'bbox_overfit'} else 4
# The high-resolution grid is needed only for the isolated 32-row bbox probe.
# Diagnostic/mini process before+after images over 5k rows; 448px fits a T4.
if STAGE in {'diagnostic', 'mini'}:
    cfg['backbone']['min_pixels'] = 50176
    cfg['backbone']['max_pixels'] = 200704

assert cfg['train']['controlled_experiment_tag'] == 'recovery_v2_7'
assert cfg['train']['quality_selection_rule'] == 'all_gates_then_outcome_mcc'
assert cfg['data']['strict_bbox_geometry']
assert cfg['data']['invalid_bbox_policy'] == 'mask'
assert cfg['model']['bbox_parameterization'] == 'cxcywh'
assert cfg['model']['bbox_grounding_mode'] == 'coordinate_softargmax'
assert cfg['model']['bbox_fp32_grounding'] is True
assert cfg['model']['bbox_size_parameterization'] == 'log_space'
assert cfg['model']['bbox_attention_dropout'] == 0.0
assert cfg['loss']['bbox_loss'] == 'detr_log_size_giou'
assert cfg['loss']['bbox_l1_ratio'] == 5.0 and cfg['loss']['bbox_giou_ratio'] == 2.0
assert cfg['loss']['bbox_attention_ratio'] == 1.0
assert cfg['train']['bbox_overfit_fp32_grounding'] is True
assert cfg['train']['bbox_overfit_disable_dropout'] is True
assert cfg['train']['bbox_overfit_require_finite_gradients'] is True
OVERFIT_STEPS = int(cfg['train']['bbox_overfit_steps'])
INTERMEDIATE_EVAL_STEPS = sorted(int(step) for step in cfg['train']['bbox_overfit_full_eval_steps'])
assert OVERFIT_STEPS > 0 and OVERFIT_STEPS % 10 == 0
assert all(0 < step < OVERFIT_STEPS for step in INTERMEDIATE_EVAL_STEPS)
EXPECTED_FULL_EVAL_STEPS = [0, *INTERMEDIATE_EVAL_STEPS, OVERFIT_STEPS]
EXPECTED_OPTIMIZER_TRACE_ROWS = OVERFIT_STEPS // 10
print('stage:', STAGE, '| epochs:', EPOCHS, '| max_pixels:', cfg['backbone']['max_pixels'])
print('selection rule:', cfg['train']['quality_selection_rule'])
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 4. Always audit complete train/validation geometry before a training gate.
from web_agent.train.gold_stages import run_gold_bbox_audit

bbox_audit = run_gold_bbox_audit(cfg)
audit_path = Path('/kaggle/working/gold_recovery_v2_7_bbox_audit.json')
audit_path.write_text(json.dumps(bbox_audit, indent=2), encoding='utf-8')
print(json.dumps(bbox_audit, indent=2))
print('BBox audit:', audit_path)
assert bbox_audit['status'] == 'PASS', 'Audit found a fatal image/file problem or no usable bbox supervision; inspect the saved report.'
assert bbox_audit['test_rows_read'] == 0

In [ ]:
# 5. Run the selected gate. No stage opens split_test.json.
from IPython.display import Image as NotebookImage, display
from web_agent.train.gold_stages import (
    build_processor,
    run_gold_bbox_overfit,
    run_gold_mini,
    run_gold_smoke,
    save_gold_bbox_probe_montage,
)
from web_agent.utils.results import save_mini_diagnostics_json, save_mini_result_csv

processor = None
montage_path = Path('/kaggle/working/gold_recovery_v2_7_bbox_probe_montage.jpg')
montage_report = None
if STAGE in {'smoke', 'bbox_overfit'}:
    montage_report = save_gold_bbox_probe_montage(cfg, montage_path, rows=32, seed=SEED)
    display(NotebookImage(filename=str(montage_path)))
    print('Review every green target box in:', montage_path)
if STAGE == 'audit':
    stage_report = bbox_audit
else:
    processor = build_processor(cfg)
    if STAGE == 'smoke':
        stage_report = run_gold_smoke(cfg, processor=processor, rows=16, seed=SEED)
    elif STAGE == 'bbox_overfit':
        assert BBOX_MONTAGE_REVIEWED is True, 'Review all 32 green boxes, then set BBOX_MONTAGE_REVIEWED=True.'
        stage_report = run_gold_bbox_overfit(cfg, processor=processor, seed=SEED)
    else:
        stage_report = run_gold_mini(
            cfg, processor=processor, train_rows=TRAIN_ROWS, val_rows=VAL_ROWS,
            epochs=EPOCHS, seed=SEED,
        )
report_path = Path(f'/kaggle/working/gold_recovery_v2_7_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = diagnostics_path = None
if STAGE in {'diagnostic', 'mini'}:
    result_csv_path = save_mini_result_csv(
        stage_report, f'/kaggle/working/gold_recovery_v2_7_{STAGE}_result.csv',
    )
    diagnostics_path = save_mini_diagnostics_json(
        stage_report, f'/kaggle/working/gold_recovery_v2_7_{STAGE}_diagnostics.json',
    )
print(json.dumps(stage_report, indent=2))
print('Report:', report_path, '| CSV:', result_csv_path, '| diagnostics:', diagnostics_path)

In [ ]:
# 6. Enforce the stage contract and the corrected v2.7 selector.
assert stage_report['status'] == 'PASS'
assert stage_report['test_rows_read'] == 0
if STAGE == 'audit':
    print('AUDIT ACCEPTED:', stage_report['training_disposition'])
    print('Retained records:', stage_report['retained_records'], '| valid bbox targets:', stage_report['bbox_supervision_rows'], '| masked bbox targets:', stage_report['masked_bbox_rows'])
    print('Restart, set STAGE to smoke, then Run All.')
elif STAGE == 'smoke':
    for name in ('bbox', 'needs_recovery', 'strategy', 'recovery_outcome', 'grounding_adapter', 'coordinate_projection', 'grounding_attention'):
        assert stage_report['probe_gradient_norms'][name] > 0, f'No gradient: {name}'
        assert stage_report['probe_update_norms'][name] > 0, f'No update: {name}'
    assert stage_report['bbox_supervised_rows'] > 0
    assert min(stage_report['spatial_tokens_per_row']) > 0
    print('SMOKE PASSED. Review all 32 montage boxes before any bbox-overfit rerun.')
elif STAGE == 'bbox_overfit':
    assert montage_report['record_ids'] == stage_report['selected_record_ids']
    assert stage_report['pre_action_feasibility']['status'] == 'PASS'
    assert stage_report['pre_action_feasibility']['conflicting_rows'] == 0
    assert all(stage_report['checks'].values())
    assert stage_report['nonfinite_gradient_steps'] == 0
    assert stage_report['fp32_grounding'] is True and stage_report['dropout_disabled'] is True
    assert stage_report['bbox_log_size_prior']['source'] == 'valid training bbox rows only'
    assert stage_report['checks']['decoded_sizes_above_registered_minimum']
    assert stage_report['checks']['attention_kl_decreased']
    assert stage_report['steps'] == OVERFIT_STEPS
    assert len(stage_report['optimizer_trace']) == EXPECTED_OPTIMIZER_TRACE_ROWS
    trajectory_steps = [row['optimizer_step'] for row in stage_report['full_set_trajectory']]
    assert trajectory_steps == EXPECTED_FULL_EVAL_STEPS, trajectory_steps
    assert all(len(row['per_row']) == stage_report['rows'] for row in stage_report['full_set_trajectory'])
    print('BBOX OVERFIT PASSED:', stage_report['initial']['bbox_mean_iou'], '->', stage_report['final']['bbox_mean_iou'])
    print('Attention KL:', stage_report['initial']['bbox_attention_kl'], '->', stage_report['final']['bbox_attention_kl'])
    print('Worst final rows:', json.dumps(stage_report['final']['worst_rows'], indent=2))
    print('Restart, set STAGE to diagnostic, then Run All.')
else:
    assert len(stage_report['history']) == EPOCHS
    assert stage_report['checkpoint_roundtrip'] is True
    assert result_csv_path.is_file() and diagnostics_path.is_file()
    last = stage_report['history'][-1]
    print('bbox:', last['bbox_mean_iou'], last['bbox_recall_iou50'])
    print('bbox centre/log-size/GIoU/attention-KL:', last['train_raw_bbox_center_l1_loss'], last['train_raw_bbox_log_size_smooth_l1_loss'], last['train_raw_bbox_giou_loss'], last['train_raw_bbox_attention_kl_loss'])
    print('needs recovery:', last['needs_recovery_macro_f1'], 'strategy:', last['strategy_attempted_macro_f1'])
    if STAGE == 'diagnostic':
        required = (
            'needs_recovery_macro_f1_beats_majority_by_0_03',
            'attempted_strategy_macro_f1_beats_majority_by_0_03',
            'transition_recovery_outcome_mcc_improves_v14_by_0_01',
            'bbox_mean_iou_at_least_0_05',
            'bbox_recall_iou50_at_least_0_01',
            'outcome_ece_not_up_more_than_0_03',
        )
        checks = stage_report['quality_gates']['checks']
        assert all(checks[name] for name in required), {name: checks[name] for name in required}
        print('DIAGNOSTIC FUNCTIONAL GATES PASSED. Restart, set STAGE to mini, then Run All.')
    else:
        assert stage_report['loss_decreased'] is True
        assert len(stage_report['epoch_checkpoints']) == EPOCHS
        assert stage_report['selection_rule'] == 'all_gates_then_outcome_mcc'
        quality = stage_report['quality_gates']
        assert quality['status'] == 'PASS'
        assert quality['selection_rule'] == stage_report['selection_rule']
        assert quality['eligible_epochs'], 'No epoch passed every registered quality gate.'
        selected_epoch = int(stage_report['selected_epoch'])
        eligible_rows = [row for row in stage_report['history'] if int(row['epoch']) in quality['eligible_epochs']]
        expected = max(eligible_rows, key=lambda row: (float(row['outcome_mcc']), -int(row['epoch'])))
        assert selected_epoch == int(expected['epoch'])
        assert quality['selected_epoch'] == selected_epoch
        assert quality['selected_epoch_is_eligible'] is True
        assert stage_report['best_checkpoint'] == stage_report['epoch_checkpoints'][str(selected_epoch)]
        assert quality['selected_checkpoint'] == stage_report['best_checkpoint']
        assert abs(float(stage_report['best_metric']) - float(expected['outcome_mcc'])) < 1e-12
        unconstrained = next(row for row in stage_report['history'] if int(row['epoch']) == int(quality['unconstrained_outcome_epoch']))
        assert abs(float(stage_report['unconstrained_best_metric']) - float(unconstrained['outcome_mcc'])) < 1e-12
        with result_csv_path.open(encoding='utf-8', newline='') as handle:
            csv_rows = list(csv.DictReader(handle))
        selected_csv_rows = [row for row in csv_rows if row['is_selected'] == 'True']
        assert len(selected_csv_rows) == 1
        assert int(selected_csv_rows[0]['epoch']) == selected_epoch
        print('V2.7 FIVE-EPOCH MINI PASSED')
        print('eligible epochs:', quality['eligible_epochs'])
        print('selected epoch:', selected_epoch, '| outcome MCC:', stage_report['best_metric'])
        print('selected checkpoint round-trip:', stage_report['best_checkpoint'])
        print('Preserve the report, CSV, diagnostics, environment, and selected checkpoint.')

## Decision rule

Audit failures are reported, never silently repaired. Invalid bbox targets may be masked only under the registered policy; unreadable images remain fatal. Smoke proves the changed heads receive gradients and updates. The bbox-overfit probe derives its 200-step trace and full-set evaluation checkpoints from configuration, preventing the stale 100-step assertion that remained in the old notebook. Diagnostic enforces the registered functional checks. Mini passes only when at least one epoch satisfies every quality gate; outcome MCC ranks those eligible epochs, the selected checkpoint is reloaded, and its prediction round-trip is verified. A v2.7 mini pass is still a development gate, not a test-set or 90-percent claim.